In [1]:
from pathlib import Path
import os
import subprocess
import sys

from google.colab import drive


def run(cmd, cwd=None, env=None):
    print("Running:", " ".join(map(str, cmd)))
    proc = subprocess.Popen(
        cmd,
        cwd=cwd,
        env=env,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )
    assert proc.stdout is not None
    for line in proc.stdout:
        print(line, end="")
    code = proc.wait()
    if code:
        raise RuntimeError(f"Command failed with exit code {code}: " + " ".join(map(str, cmd)))


drive.mount("/content/drive")

WORKDIR = Path("/content/drive/MyDrive/Research/SSM_World_Models")
R2DREAMER_DIR = WORKDIR / "r2dreamer"
TDMPC2_DIR = WORKDIR / "tdmpc2"
DATA_DIR = WORKDIR / "data" / "dmc_expert"

R2DREAMER_REPO = "https://github.com/hanswalker/r2dreamer.git"
R2DREAMER_BRANCH = "mamba3-rssm-experiment"
TDMPC2_REPO = "https://github.com/nicklashansen/tdmpc2.git"
MAMBA_REPO = "git+https://github.com/state-spaces/mamba.git"

WORKDIR.mkdir(parents=True, exist_ok=True)
DATA_DIR.mkdir(parents=True, exist_ok=True)

if not R2DREAMER_DIR.exists():
    run(["git", "clone", "--branch", R2DREAMER_BRANCH, R2DREAMER_REPO, str(R2DREAMER_DIR)])
else:
    remotes = subprocess.run(
        ["git", "remote"], cwd=R2DREAMER_DIR, check=True, capture_output=True, text=True
    ).stdout.split()
    if "notebook-source" not in remotes:
        run(["git", "remote", "add", "notebook-source", R2DREAMER_REPO], cwd=R2DREAMER_DIR)
    else:
        run(["git", "remote", "set-url", "notebook-source", R2DREAMER_REPO], cwd=R2DREAMER_DIR)
    run(["git", "fetch", "notebook-source", R2DREAMER_BRANCH], cwd=R2DREAMER_DIR)
    run(["git", "checkout", "-B", R2DREAMER_BRANCH, f"notebook-source/{R2DREAMER_BRANCH}"], cwd=R2DREAMER_DIR)
    run(["git", "reset", "--hard", f"notebook-source/{R2DREAMER_BRANCH}"], cwd=R2DREAMER_DIR)

if not TDMPC2_DIR.exists():
    run(["git", "clone", TDMPC2_REPO, str(TDMPC2_DIR)])
else:
    run(["git", "fetch", "origin"], cwd=TDMPC2_DIR)
    run(["git", "reset", "--hard", "origin/main"], cwd=TDMPC2_DIR)

run([
    sys.executable,
    "-m",
    "pip",
    "install",
    "-q",
    "numpy==2.0.2",
    "pyyaml",
    "zarr<3",
    "huggingface_hub",
    "dm_control==1.0.28",
    "mujoco==3.3.0",
    "omegaconf",
    "hydra-core",
    "tensorboard>=2.20,<3",
    "gymnasium==1.2.0",
    "tensordict",
    "torchrl",
    "kornia",
    "termcolor",
    "tqdm",
    "pandas",
    "moviepy",
    "imageio",
    "imageio-ffmpeg",
    "h5py",
    "wheel",
    "ninja",
    "packaging",
    "einops",
])

# Mamba3 tries optional TileLang kernels during import. In Colab this path can
# fail inside NumPy before Mamba gets a chance to fall back, and we do not need
# TileLang for the SISO Mamba3 config used here.
run([sys.executable, "-m", "pip", "uninstall", "-y", "tilelang"])


def clear_mamba_modules():
    for name in list(sys.modules):
        if name == "mamba_ssm" or name.startswith("mamba_ssm."):
            del sys.modules[name]


try:
    clear_mamba_modules()
    from mamba_ssm.modules.mamba3 import Mamba3
except Exception:
    build_env = dict(os.environ)
    build_env["MAMBA_FORCE_BUILD"] = "TRUE"
    run([
        sys.executable,
        "-m",
        "pip",
        "install",
        "--no-cache-dir",
        MAMBA_REPO,
        "--no-build-isolation",
    ], env=build_env)
    run([sys.executable, "-m", "pip", "uninstall", "-y", "tilelang"])
    clear_mamba_modules()
    from mamba_ssm.modules.mamba3 import Mamba3

os.environ["MUJOCO_GL"] = "egl"
os.environ["PYOPENGL_PLATFORM"] = "egl"
os.environ["TDMPC2_DIR"] = str(TDMPC2_DIR)
os.environ["DMC_EXPERT_DATA_DIR"] = str(DATA_DIR)

os.chdir(R2DREAMER_DIR)
if str(R2DREAMER_DIR) not in sys.path:
    sys.path.insert(0, str(R2DREAMER_DIR))

import numpy as np
import torch

print("R2Dreamer:", R2DREAMER_DIR)
print("TD-MPC2:", TDMPC2_DIR)
print("Data:", DATA_DIR)
print("NumPy:", np.__version__)
print("Mamba3:", Mamba3)
print("CUDA:", torch.cuda.is_available())

if not torch.cuda.is_available():
    raise RuntimeError("Use a Colab GPU runtime before running this notebook.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Running: git remote set-url notebook-source https://github.com/hanswalker/r2dreamer.git
Running: git fetch notebook-source mamba3-rssm-experiment
From https://github.com/hanswalker/r2dreamer
 * branch            mamba3-rssm-experiment -> FETCH_HEAD
   8a1ec82..3167de2  mamba3-rssm-experiment -> notebook-source/mamba3-rssm-experiment
Running: git checkout -B mamba3-rssm-experiment notebook-source/mamba3-rssm-experiment
Reset branch 'mamba3-rssm-experiment'
M	runs/atari.sh
M	runs/crafter.sh
M	runs/dmc.sh
M	runs/dmc_subtle.sh
M	runs/isaaclab.sh
M	runs/memorymaze.sh
M	runs/metaworld.sh
Branch 'mamba3-rssm-experiment' set up to track remote branch 'mamba3-rssm-experiment' from 'notebook-source'.
Your branch is up to date with 'notebook-source/mamba3-rssm-experiment'.
Running: git reset --hard notebook-source/mamba3-rssm-experiment
HEAD is now at 3167de2 model chan

In [ ]:
import subprocess
import sys
import time
import textwrap

import zarr
from IPython.display import clear_output

COLLECTOR = R2DREAMER_DIR / "scripts" / "collect_dmc_expert_data.py"

parallel_tasks = [
    "walker/walk",       # resumes the default walker dataset in DATA_DIR
    "cartpole/swingup",
    "cheetah/run",
    "finger/spin",
    "reacher/easy",
]

NUM_EPISODES = 500
CHECKPOINT_SEED = 1
SEED = 1
ACTION_REPEAT = 2
MAX_EPISODE_STEPS = 1000
SAVE_IMAGES = True
IMAGE_SIZE = 64
REFRESH_SECONDS = 10
RECENT_LOG_LINES = 25
PROGRESS_EVERY = 25

PARALLEL_ROOT = DATA_DIR / "parallel_runs"
CONFIG_DIR = PARALLEL_ROOT / "configs"
LOG_DIR = PARALLEL_ROOT / "logs"
CONFIG_DIR.mkdir(parents=True, exist_ok=True)
LOG_DIR.mkdir(parents=True, exist_ok=True)


def task_store_name(task):
    if task == "ball_in_cup/catch":
        return "cup_catch.zarr"
    return task.replace("/", "_").replace("-", "_").replace("__", "_") + ".zarr"


def read_progress(out_dir, task):
    store_path = out_dir / task_store_name(task)
    if not store_path.exists():
        return 0, 0, "not created"
    try:
        root = zarr.open(str(store_path), mode="r")
        episodes = int(root["episode_length"].shape[0]) if "episode_length" in root else 0
        rows = int(root["obs"].shape[0]) if "obs" in root else 0
        return episodes, rows, str(store_path)
    except Exception as exc:
        return 0, 0, f"read pending: {type(exc).__name__}"


def read_new_log_lines(log_path, offset):
    if not log_path.exists():
        return offset, []
    with log_path.open("r", encoding="utf-8", errors="replace") as f:
        f.seek(offset)
        lines = f.readlines()
        offset = f.tell()
    return offset, lines


processes = []
log_offsets = {}
recent_lines = []

for task in parallel_tasks:
    slug = task.replace("/", "_")

    # Keep walker/walk in DATA_DIR so training can use DATA_DIR / "walker_walk.zarr".
    # Other tasks go in separate folders to avoid manifest write races.
    out_dir = DATA_DIR if task == "walker/walk" else PARALLEL_ROOT / slug
    out_dir.mkdir(parents=True, exist_ok=True)

    config_path = CONFIG_DIR / f"{slug}.yaml"
    log_path = LOG_DIR / f"{slug}.log"

    config_path.write_text(
        textwrap.dedent(f"""
        tdmpc2_root: {TDMPC2_DIR}
        output_dir: {out_dir}

        tasks:
          - {task}

        num_episodes: {NUM_EPISODES}
        checkpoint_seed: {CHECKPOINT_SEED}
        seed: {SEED}

        action_repeat: {ACTION_REPEAT}
        max_episode_steps: {MAX_EPISODE_STEPS}

        save_images: {str(SAVE_IMAGES).lower()}
        image_size: {IMAGE_SIZE}

        resume: true
        progress_every: {PROGRESS_EVERY}
        """).strip() + "\n",
        encoding="utf-8",
    )

    log_file = log_path.open("w", buffering=1, encoding="utf-8")
    env = dict(os.environ)
    env["PYTHONUNBUFFERED"] = "1"
    proc = subprocess.Popen(
        [sys.executable, "-u", str(COLLECTOR), "--config", str(config_path)],
        stdout=log_file,
        stderr=subprocess.STDOUT,
        text=True,
        env=env,
    )

    processes.append((task, proc, log_file, log_path, out_dir))
    log_offsets[task] = 0
    recent_lines.append(f"[{task}] started pid={proc.pid}, output={out_dir}, log={log_path}")

while True:
    all_done = True
    statuses = []

    for task, proc, _, log_path, out_dir in processes:
        log_offsets[task], lines = read_new_log_lines(log_path, log_offsets[task])
        for line in lines:
            line = line.rstrip()
            if line:
                recent_lines.append(f"[{task}] {line}")
        recent_lines = recent_lines[-RECENT_LOG_LINES:]

        code = proc.poll()
        if code is None:
            all_done = False
            state = "running"
        elif code == 0:
            state = "done"
        else:
            state = f"failed({code})"

        episodes, rows, store = read_progress(out_dir, task)
        statuses.append((task, state, episodes, rows, store, log_path))

    clear_output(wait=True)
    print(f"Live collector progress | refresh={REFRESH_SECONDS}s | target={NUM_EPISODES} episodes per task | log every {PROGRESS_EVERY}")
    print("-" * 120)
    print(f"{'task':<22} {'state':<12} {'episodes':>10} {'rows':>10}  store/log")
    for task, state, episodes, rows, store, log_path in statuses:
        print(f"{task:<22} {state:<12} {episodes:>4}/{NUM_EPISODES:<5} {rows:>10}  {store}")
        print(f"{'':<48} log: {log_path}")

    print("\nRecent log lines")
    print("-" * 120)
    print("\n".join(recent_lines[-RECENT_LOG_LINES:]) if recent_lines else "No log lines yet.")

    if all_done:
        break
    time.sleep(REFRESH_SECONDS)

failed = []
for task, proc, log_file, log_path, _ in processes:
    log_file.close()
    if proc.returncode != 0:
        failed.append((task, log_path))

if failed:
    print("\nFailures:")
    for task, log_path in failed:
        print(f"\n--- {task} last log lines ---")
        lines = log_path.read_text(errors="replace").splitlines()
        print("\n".join(lines[-120:]))
    raise RuntimeError(f"{len(failed)} collectors failed")

print("\nAll collectors finished successfully.")

In [3]:
import json
import os
import subprocess
import sys
import time

from IPython.display import clear_output

TRAIN_STORE = DATA_DIR / "walker_walk.zarr"
if not TRAIN_STORE.exists():
    raise FileNotFoundError(f"Missing training dataset: {TRAIN_STORE}")

RUN_TAG = time.strftime("%Y%m%d_%H%M%S")

TRAIN_RUNS = [
    {
        "name": "gru",
        "model": "size_small_gru",
        "logdir": WORKDIR / "runs" / f"offline_size_small_gru_walker_walk_fresh_{RUN_TAG}",
    },
    {
        "name": "mamba3",
        "model": "size_small_mamba3",
        "logdir": WORKDIR / "runs" / f"offline_size_small_mamba3_dstate32_headdim64_walker_walk_fresh_{RUN_TAG}",
    },
]

REFRESH_SECONDS = 10
RECENT_LOG_LINES = 30
processes = []
log_offsets = {}
recent_lines = []

for run_cfg in TRAIN_RUNS:
    name = run_cfg["name"]
    model = run_cfg["model"]
    logdir = run_cfg["logdir"]
    logdir.mkdir(parents=True, exist_ok=True)
    stdout_log = logdir / "notebook_stdout.log"

    cmd = [
        sys.executable,
        "-u",
        "train.py",
        "--config-name",
        "offline_dmc_expert",
        f"model={model}",
        f"offline.data_path={TRAIN_STORE}",
        "env.task=dmc_walker_walk",
        "offline.resume=false",
        f"logdir={logdir}",
    ]

    log_file = stdout_log.open("w", buffering=1, encoding="utf-8")
    env = dict(os.environ)
    env["PYTHONUNBUFFERED"] = "1"
    proc = subprocess.Popen(
        cmd,
        cwd=R2DREAMER_DIR,
        stdout=log_file,
        stderr=subprocess.STDOUT,
        text=True,
        env=env,
    )

    processes.append((name, proc, log_file, stdout_log, logdir))
    log_offsets[name] = 0
    recent_lines.append(f"[{name}] started pid={proc.pid}, logdir={logdir}, stdout={stdout_log}")


def read_new_log_lines(log_path, offset):
    if not log_path.exists():
        return offset, []
    with log_path.open("r", encoding="utf-8", errors="replace") as f:
        f.seek(offset)
        lines = f.readlines()
        offset = f.tell()
    return offset, lines


def fmt_metric(value):
    if value is None:
        return "-"
    try:
        return f"{float(value):.3g}"
    except Exception:
        return str(value)


def read_metric_summary(logdir):
    metrics_path = logdir / "metrics.jsonl"
    update = None
    opt_loss = None
    eval_score = None
    if not metrics_path.exists():
        return "-", "-", "-"
    try:
        with metrics_path.open("r", encoding="utf-8", errors="replace") as f:
            for line in f:
                line = line.strip()
                if not line:
                    continue
                row = json.loads(line)
                update = row.get("train/opt/updates", update)
                opt_loss = row.get("train/opt/loss", opt_loss)
                eval_score = row.get("episode/eval_score", eval_score)
    except Exception as exc:
        return "read_err", type(exc).__name__, "-"
    return fmt_metric(update), fmt_metric(opt_loss), fmt_metric(eval_score)


while True:
    all_done = True
    statuses = []

    for name, proc, _, stdout_log, logdir in processes:
        log_offsets[name], lines = read_new_log_lines(stdout_log, log_offsets[name])
        for line in lines:
            line = line.rstrip()
            if line:
                recent_lines.append(f"[{name}] {line}")
        recent_lines = recent_lines[-RECENT_LOG_LINES:]

        code = proc.poll()
        if code is None:
            all_done = False
            state = "running"
        elif code == 0:
            state = "done"
        else:
            state = f"failed({code})"

        latest_ckpt = logdir / "latest.pt"
        best_ckpt = logdir / "best.pt"
        update, opt_loss, eval_score = read_metric_summary(logdir)
        statuses.append((name, state, update, opt_loss, eval_score, latest_ckpt.exists(), best_ckpt.exists(), stdout_log))

    clear_output(wait=True)
    print(f"Live training progress | refresh={REFRESH_SECONDS}s")
    print("-" * 120)
    print(f"{'model':<10} {'state':<12} {'update':>8} {'opt_loss':>10} {'eval':>10} {'latest':<7} {'best':<5} stdout")
    for name, state, update, opt_loss, eval_score, has_latest, has_best, stdout_log in statuses:
        print(f"{name:<10} {state:<12} {update:>8} {opt_loss:>10} {eval_score:>10} {str(has_latest):<7} {str(has_best):<5} {stdout_log}")

    print("\nRecent log lines")
    print("-" * 120)
    print("\n".join(recent_lines[-RECENT_LOG_LINES:]) if recent_lines else "No log lines yet.")

    if all_done:
        break
    time.sleep(REFRESH_SECONDS)

failed = []
for name, proc, log_file, stdout_log, _ in processes:
    log_file.close()
    if proc.returncode != 0:
        failed.append((name, stdout_log))

if failed:
    print("\nFailures:")
    for name, stdout_log in failed:
        print(f"\n--- {name} last log lines ---")
        lines = stdout_log.read_text(errors="replace").splitlines()
        print("\n".join(lines[-120:]))
    raise RuntimeError(f"{len(failed)} training runs failed")

print("\nBoth training runs finished successfully.")

Live training progress | refresh=10s
------------------------------------------------------------------------------------------------------------------------
model      state          update   opt_loss       eval latest  best  stdout
gru        running             -          -          - False   False /content/drive/MyDrive/Research/SSM_World_Models/runs/offline_size_small_gru_walker_walk_fresh_20260615_222728/notebook_stdout.log
mamba3     failed(1)           -          -          - False   False /content/drive/MyDrive/Research/SSM_World_Models/runs/offline_size_small_mamba3_dstate32_headdim64_walker_walk_fresh_20260615_222728/notebook_stdout.log

Recent log lines
------------------------------------------------------------------------------------------------------------------------
[mamba3]     post = self.rssm.observe(embed, data["action"], initial, data["is_first"])
[mamba3]            ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
[mamba3]   File "/content/dri

KeyboardInterrupt: 